# Librerías y configuración inicial

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Carga de datos

In [2]:
current = Path.cwd()
file_path = current.parent / "data" / "preprocessed_data" / "preprocess_data.parquet"
if not file_path.exists():
    raise FileNotFoundError(f"[ERROR] No se encontró el archivo: {file_path}")

df = pd.read_parquet(file_path)
df = df.sort_values("datetime") # Ordenamos por si acaso

# Generación de variables temporales y exógenas

## Variables temporales cíclicas

In [3]:
# Capturamos la continuidad cíclica seno-coseno entre los extremos (evitamos discontinuidad 23 -> 0, domingo->lunes, diciembre->enero)
df['hora_sin'] = np.sin(2*np.pi*df['datetime'].dt.hour/24)
df['hora_cos'] = np.cos(2*np.pi*df['datetime'].dt.hour/24)
df["day_sin"]  = np.sin(2*np.pi*df['datetime'].dt.dayofweek/7)
df["day_cos"]  = np.cos(2*np.pi*df['datetime'].dt.dayofweek/7)
df['mes_sin']  = np.sin(2*np.pi*df['datetime'].dt.month/12)
df['mes_cos']  = np.cos(2*np.pi*df['datetime'].dt.month/12)

## Lags y rolling statistics

In [4]:
# Memoria temporal según lo observado en AFC y PACF
lags = [1, 2, 3, 6, 12, 24, 48, 72, 168]

for lag in lags:
    df[f'demanda_lag_{lag}'] = df['demanda'].shift(lag)
    df[f'precio_lag_{lag}']  = df['precio'].shift(lag)

In [5]:
# Tendencias a 1 día, 2 días y 7 días
windows = [24, 48, 168]

for w in windows:
    df[f'demanda_roll_mean_{w}'] = df['demanda'].shift(1).rolling(w).mean()
    df[f'precio_roll_mean_{w}']  = df['precio'].shift(1).rolling(w).mean()
    df[f'demanda_roll_std_{w}']  = df['demanda'].shift(1).rolling(w).std()
    df[f'precio_roll_std_{w}']   = df['precio'].shift(1).rolling(w).std()

## Contexto energético del sistema

In [6]:
# Producción dependiente de condiciones meteorológicas y no programable
df["renovables_no_gestionables"] = df["gen_eolica"] + df["gen_solar_fotovoltaica"] + df["gen_solar_termica"]

# Producción renovable regulable o parcialmente programable que aporta estabilidad al sistema
df["renovables_gestionables"] = df["gen_hidraulica"] + df["gen_otras_renovables"] + df["gen_residuos_renovables"]

# Generación no renovable
df["gen_no_renovable_total"] = (
    df["gen_carbon"] + df["gen_ciclo_combinado"] + df["gen_cogeneracion"] + df["gen_fuel_gas"]  + df["gen_nuclear"] + df["gen_residuos_no_renovables"] + df["gen_turbina_vapor"]
)

# Ratio renovable
df["ratio_renovable"] = (
    (df["renovables_no_gestionables"] + df["renovables_gestionables"]) / (df["renovables_no_gestionables"] + df["renovables_gestionables"] + df["gen_no_renovable_total"])
)

# Bombeo neto
df["bombeo_net"] = df["alm_turbinacion_bombeo"] - df["alm_consumo_bombeo"]

## Intercambios internacionales

In [7]:
# Eliminamos Andorra por poco peso
df["import_total"] = df["francia_import"] + df["portugal_import"] + df["marruecos_import"]
df["export_total"] = df["francia_export"] + df["portugal_export"] + df["marruecos_export"]

# Selección de variables y creación de dataset final

In [ ]:
feature_cols = [
    # Variables temporales cíclicas
    'hora_sin', 'hora_cos', 'day_sin', 'day_cos', 'mes_sin', 'mes_cos',
    
    # Lags
    'demanda_lag_1', 'demanda_lag_2', 'demanda_lag_3', 'demanda_lag_6', 'demanda_lag_24', 'demanda_lag_48', 'demanda_lag_72', 'demanda_lag_168',
    'precio_lag_1', 'precio_lag_2', 'precio_lag_3', 'precio_lag_6', 'precio_lag_24', 'precio_lag_48', 'precio_lag_72', 'precio_lag_168',
    
    # Rollings
    'demanda_roll_mean_24', 'demanda_roll_mean_48', 'demanda_roll_mean_168', 'demanda_roll_std_24',
    'precio_roll_mean_24', 'precio_roll_mean_48', 'precio_roll_mean_168', 'precio_roll_std_24',

    # Contexto energético
    'renovables_no_gestionables', 'renovables_gestionables', 'gen_no_renovable_total', 'ratio_renovable', 'bombeo_net',
    
    # Tecnologías clave
    'gen_ciclo_combinado', 'gen_eolica', 'gen_solar_fotovoltaica', 'gen_hidraulica', 'gen_otras_renovables', 'gen_residuos_renovables',

    # Intercambios internacionales
    'import_total', 'export_total'
]

# Variables objetivo
target_cols = ['demanda', 'precio']

df_model = df[['datetime', *target_cols, *feature_cols]].dropna().copy() # Se eliminarán 168 filas por los nulo generados por los lags

print(f"Dimensiones del dataset de modelado:\n{df_model.shape}")
print(f"Columnas del dataset de modelado:\n{df_model.columns}")

Dimensiones del dataset de modelado:
(61200, 46)
Columnas del dataset de modelado:
Index(['datetime', 'demanda', 'precio', 'hora_sin', 'hora_cos', 'day_sin',
       'day_cos', 'mes_sin', 'mes_cos', 'demanda_lag_1', 'demanda_lag_2',
       'demanda_lag_3', 'demanda_lag_6', 'demanda_lag_24', 'demanda_lag_48',
       'demanda_lag_72', 'demanda_lag_168', 'precio_lag_1', 'precio_lag_2',
       'precio_lag_3', 'precio_lag_6', 'precio_lag_24', 'precio_lag_48',
       'precio_lag_72', 'precio_lag_168', 'demanda_roll_mean_24',
       'demanda_roll_mean_48', 'demanda_roll_mean_168', 'demanda_roll_std_24',
       'precio_roll_mean_24', 'precio_roll_mean_48', 'precio_roll_mean_168',
       'precio_roll_std_24', 'renovables_no_gestionables',
       'renovables_gestionables', 'gen_no_renovable_total', 'ratio_renovable',
       'bombeo_net', 'gen_ciclo_combinado', 'gen_eolica',
       'gen_solar_fotovoltaica', 'gen_hidraulica', 'gen_otras_renovables',
       'gen_residuos_renovables', 'import_total',

In [9]:
current = Path.cwd()
path = current.parent / "data" / "model_data"
path.mkdir(parents=True, exist_ok=True)

file_path = path / "model_data.parquet"
df_model.to_parquet(file_path)